# 🔐 Lab: Object-Oriented Design in Cybersecurity Systems
**Course:** IST1012 
**Duration:** 50 Minutes  
**Topics Covered:** Inheritance · Polymorphism · Composition · Interfaces (Abstract Classes)

---

> **Scenario:** You are a junior developer at a cybersecurity firm.
> Your team is building a **Threat Detection Framework** — a system that handles
> different types of security alerts, scanners, and response actions.
> You will use OOP principles to design it cleanly and professionally.

---

## 🎯 Learning Objectives

By the end of this lab, you will be able to:

1. **Define and extend** classes using **inheritance** to model security entities
2. **Apply polymorphism** to handle different threat types with a unified interface
3. **Use composition** to build complex security objects from smaller components
4. **Implement abstract base classes** to enforce structure across a security framework
5. **Distinguish** when to use inheritance vs. composition in real system design

---

### 🔧 Warm-Up Code

Run the cell below. Read it carefully — this is the foundation you'll build on.

In [1]:
# Run this cell to verify your Python environment
import sys
print("Python version:", sys.version)

# Quick class recall
class SecurityEvent(object):
    def __init__(self, event_id, severity):
        self.event_id = event_id
        self.severity = severity

    def describe(self):
        return "Event " + str(self.event_id) + " | Severity: " + self.severity

e = SecurityEvent("EVT-001", "HIGH")
print(e.describe())

Python version: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Event EVT-001 | Severity: HIGH


**Warm-Up Task:** In the cell below, create a second `SecurityEvent` object with:
- `event_id` = `"EVT-002"`
- `severity` = `"LOW"`

Then print its description.

In [2]:
test = SecurityEvent("EVT-002", "LOW")
print(test.describe())


Event EVT-002 | Severity: LOW


---

## 📖 Section 2 – Core Concepts (~ 10 minutes)

Read each concept block and run the example code. You do **not** need to modify these cells — just understand them.

---

### 🧬 Concept 1: Inheritance

> Inheritance lets a **child class** reuse and extend the behavior of a **parent class**.
> Think of it as: *"A MalwareAlert IS-A SecurityAlert with extra features."*
> Use inheritance when there is a clear **is-a** relationship.

In [3]:
# Parent class
class SecurityAlert(object):
    def __init__(self, alert_id, severity):
        self.alert_id = alert_id
        self.severity = severity

    def describe(self):
        return "[ALERT " + self.alert_id + "] Severity: " + self.severity

# Child class — inherits from SecurityAlert
class MalwareAlert(SecurityAlert):
    def __init__(self, alert_id, severity, malware_name):
        super(MalwareAlert, self).__init__(alert_id, severity)  # call parent constructor
        self.malware_name = malware_name

    def describe(self):
        base = super(MalwareAlert, self).describe()
        return base + " | Malware: " + self.malware_name

alert = MalwareAlert("A-001", "CRITICAL", "Ransomware.X")
print(alert.describe())

[ALERT A-001] Severity: CRITICAL | Malware: Ransomware.X


---

### 🔄 Concept 2: Polymorphism

> Polymorphism means **many forms** — the same method name behaves differently
> depending on which class is calling it.
> This lets you treat different alert types **uniformly** in your code.

In [4]:
class PhishingAlert(SecurityAlert):
    def __init__(self, alert_id, severity, sender_email):
        super(PhishingAlert, self).__init__(alert_id, severity)
        self.sender_email = sender_email

    def describe(self):
        base = super(PhishingAlert, self).describe()
        return base + " | Phishing from: " + self.sender_email

# Polymorphism in action — same loop, different describe() outputs
alerts = [
    MalwareAlert("A-001", "CRITICAL", "Ransomware.X"),
    PhishingAlert("A-002", "HIGH", "hacker@evil.com"),
    SecurityAlert("A-003", "LOW")
]

for a in alerts:
    print(a.describe())

[ALERT A-001] Severity: CRITICAL | Malware: Ransomware.X
[ALERT A-002] Severity: HIGH | Phishing from: hacker@evil.com
[ALERT A-003] Severity: LOW


---

### 🧩 Concept 3: Composition

> Composition means **building a class using other objects** as attributes.
> Think: *"A SecurityScanner HAS-A Logger and HAS-A RuleSet."*
> Use composition when there is a **has-a** relationship.

In [5]:
class Logger(object):
    def __init__(self):
        self.logs = []

    def log(self, message):
        entry = "[LOG] " + message
        self.logs.append(entry)
        print(entry)

class RuleSet(object):
    def __init__(self, rules):
        self.rules = rules

    def check(self, input_data):
        for rule in self.rules:
            if rule in input_data:
                return True   # threat detected
        return False

# SecurityScanner COMPOSES Logger and RuleSet
class SecurityScanner(object):
    def __init__(self, name, rules):
        self.name = name
        self.logger = Logger()          # HAS-A Logger
        self.ruleset = RuleSet(rules)   # HAS-A RuleSet

    def scan(self, data):
        self.logger.log(self.name + " scanning: " + data)
        if self.ruleset.check(data):
            self.logger.log("THREAT DETECTED in: " + data)
            return True
        return False

scanner = SecurityScanner("EmailScanner", ["exec(", "<script>", "DROP TABLE"])
scanner.scan("Hello world")
scanner.scan("<script>alert('xss')</script>")

[LOG] EmailScanner scanning: Hello world
[LOG] EmailScanner scanning: <script>alert('xss')</script>
[LOG] THREAT DETECTED in: <script>alert('xss')</script>


True

---

### 🏗️ Concept 4: Abstract Base Classes (Interfaces in Python)

> Python uses `abc.ABC` and `@abstractmethod` to define **abstract classes**.
> An abstract class is a **blueprint** — it defines what methods *must* exist
> in any subclass, but does not implement them.
> This enforces a **contract** across your system.

In [6]:
from abc import ABC, abstractmethod

class ThreatResponder(ABC):  # Abstract base class

    @abstractmethod
    def respond(self, alert):
        pass  # No implementation — subclasses MUST implement this

    @abstractmethod
    def log_response(self, alert):
        pass

# Concrete class that implements the interface
class EmailNotifier(ThreatResponder):
    def respond(self, alert):
        print("Sending email for: " + alert.describe())

    def log_response(self, alert):
        print("Logged email response for alert " + alert.alert_id)

notifier = EmailNotifier()
test_alert = MalwareAlert("A-010", "HIGH", "Trojan.Z")
notifier.respond(test_alert)
notifier.log_response(test_alert)

Sending email for: [ALERT A-010] Severity: HIGH | Malware: Trojan.Z
Logged email response for alert A-010


---

## 🛠️ Section 3 – Guided Exercises (~ 25 minutes)

Work through these tasks in order. Each builds on the previous.
**Do not skip ahead.** Read each task description carefully before coding.

> 💡 **Tip:** If you get stuck, re-read the concept examples above.
> Do NOT copy from them directly — understand the pattern, then apply it yourself.

---

### ✅ Exercise 1 – Inheritance (Easy)

The `SecurityAlert` class is already defined above.  
Your task: Create a new subclass called **`NetworkIntrusionAlert`** that:

- Inherits from `SecurityAlert`
- Adds a new attribute: `source_ip` (the IP address of the suspected attacker)
- Overrides `describe()` to include the source IP in its output

Example expected output:  
`[ALERT A-005] Severity: HIGH | Source IP: 192.168.1.99`

In [11]:
class NetworkIntrusionAlert(SecurityAlert):
    def __init__(self, alert_id, severity, source_ip):
        super().__init__(alert_id, severity)  # call parent constructor
        self.source_ip = source_ip

    def describe(self):
        base = super().describe()
        return base + " | Source IP: " + self.source_ip

# Test your class (do not modify these lines)
intrusion = NetworkIntrusionAlert("A-005", "HIGH", "192.168.1.99")
print(intrusion.describe())

[ALERT A-005] Severity: HIGH | Source IP: 192.168.1.99


---

### ✅ Exercise 2 – Polymorphism (Medium)

You now have at least three alert types:  
`SecurityAlert`, `MalwareAlert`, `PhishingAlert`, and your new `NetworkIntrusionAlert`.

Your task:
1. Create a **list** named `alert_log` containing at least one instance of each alert type
2. Write a function called `print_all_alerts(alert_list)` that loops through the list and prints each alert's description
3. Call the function with your list

The key insight: the function should work **without checking the type** of each alert.

In [62]:
# Exercise 2 – Polymorphic alert printer

alert_log = [
    SecurityAlert("A-001", "LOW"),
    MalwareAlert("A-002", "CRITICAL", "Ransomware.X"),
    PhishingAlert("A-003", "HIGH", "hacker@evil.com"),
    NetworkIntrusionAlert("A-004", "MEDIUM", "192.168.1.99")
]

for i in alert_log:
    print(i.describe())


[ALERT A-001] Severity: LOW
[ALERT A-002] Severity: CRITICAL | Malware: Ransomware.X
[ALERT A-003] Severity: HIGH | Phishing from: hacker@evil.com
[ALERT A-004] Severity: MEDIUM | Source IP: 192.168.1.99


---

### ✅ Exercise 3 – Composition (Medium)

You will build a **`UserAccount`** class that uses composition.

First, a helper class `PasswordPolicy` is partially written for you below — **complete it**.  
Then build `UserAccount` that **has-a** `PasswordPolicy`.

**`PasswordPolicy` requirements:**
- `__init__` takes `min_length` (int) and `require_special` (bool)
- `is_valid(password)` returns `True` if:
  - password length >= min_length
  - if `require_special` is True, password must contain at least one of: `! @ # $ %`

**`UserAccount` requirements:**
- `__init__` takes `username` and a `PasswordPolicy` object
- `set_password(password)` checks the policy and either saves it or prints an error
- `get_status()` returns a string with the username and whether a password is set

In [61]:
# Exercise 3 – Composition: PasswordPolicy + UserAccount

class PasswordPolicy(object):
    def __init__(self, min_length, require_special):
        self.min_length = min_length
        self.require_special = require_special

    def is_valid(self, password):
        if len(password) < self.min_length:
            return False

        if self.require_special:
            special_chars = "!@#$%"
            has_special = False
            for char in password:
                if char in special_chars:
                    has_special = True
            if not has_special:
                return False
        return True


class UserAccount(object):
    def __init__(self, username, policy):
        self.username = username
        self.policy = policy
        self.password = None

    def set_password(self, password):
        if self.policy.is_valid(password):
            self.password = password
            print("Password set successfully")
        else:
            print("Error: password does not meet the policy requirements")

    def get_status(self):
        if self.password is None:
            return self.username + ": password is not set"
        else:
            return self.username + ": password is set"


# Test your implementation
policy = PasswordPolicy(min_length=8, require_special=True)
account = UserAccount("alice", policy)
account.set_password("short")           # should print an error
account.set_password("strongpass")      # should print an error
account.set_password("strongpass#1")    # should succeed
print(account.get_status())

Error: password does not meet the policy requirements
Error: password does not meet the policy requirements
Password set successfully
alice: password is set


---

### ✅ Exercise 4 – Abstract Base Class (Medium-Hard)

Design an abstract class **`BaseScanner`** that acts as an interface for all scanners in your framework.

**`BaseScanner` must define these abstract methods:**
- `scan(target)` — performs the scan
- `get_report()` — returns a summary string of findings

Then implement **two concrete subclasses:**

**`PortScanner`:**
- `__init__` takes a `host` string
- `scan(target)` — `target` is a list of port numbers; mark ports as "open" if they are in `[22, 80, 443, 8080]` (simulate open ports)
- `get_report()` — returns a string listing open ports found

**`PasswordAuditScanner`:**
- `__init__` takes a `username`
- `scan(target)` — `target` is a list of passwords to test; flag any that are shorter than 8 characters or in the list `["password", "123456", "admin"]`
- `get_report()` — returns a string listing weak passwords found

In [64]:
from abc import ABC, abstractmethod

# Exercise 4 – Abstract Scanner Interface

class BaseScanner(ABC):
    @abstractmethod
    def scan(self, target):
        pass

    @abstractmethod
    def get_report(self):
        pass

class PortScanner(BaseScanner):
    def __init__(self, host):
        self.host = host
        self.allowed_ports = [22, 80, 443, 8080]
        self.open_ports = []

    def scan(self, target):
        if isinstance(target, dict):
            target = target.get("ports", [])

        self.open_ports = []
        for item in target:
            if item in self.allowed_ports:
                self.open_ports.append(item)

    def get_report(self):
        return "Open ports found: " + str(self.open_ports)

class PasswordAuditScanner(BaseScanner):
    def __init__(self, username):
        self.username = username
        self.__unallowed_passwords = ["password", "123456", "admin"]
        self.flagged_passwords = []

    def scan(self, target):
        if isinstance(target, dict):
            target = target.get("passwords", [])

        self.flagged_passwords = []
        for item in target:
            if item in self.__unallowed_passwords or len(item) < 8:
                self.flagged_passwords.append(item)

    def get_report(self):
        return "Weak passwords found: " + str(self.flagged_passwords)

---

### ✅ Exercise 5 – Putting It Together (Medium-Hard)

Now combine what you've built.  
Create a **`ThreatDetectionFramework`** class using **composition**.

It should:
- Accept a **list of scanners** (any `BaseScanner` subclasses) in `__init__`
- Have a method `run_all(target)` that runs every scanner's `scan()` with the given `target`
- Have a method `full_report()` that prints all scanners' reports

Instantiate the framework with at least one `PortScanner` and one `PasswordAuditScanner`, run it, and print the full report.

In [65]:
# Exercise 5 – ThreatDetectionFramework

class ThreatDetectionFramework():
    def __init__(self, scanners):
        self.scanners = scanners

    def run_all(self, target):
        for scanner in self.scanners:
            scanner.scan(target)

    def full_report(self):
        print("Thread Detection Framework Full Report")
        for scanner in self.scanners:
            print(scanner.get_report())


port_scanner = PortScanner("192.168.1.1")
pw_scanner = PasswordAuditScanner("bob")

test = ThreatDetectionFramework([port_scanner, pw_scanner])

target = {
    "ports": [22, 23, 80, 443, 3306, 8080, 123],
    "passwords": ["password", "hello", "Str0ng!Pass", "admin", "supersecure99"]
}

test.run_all(target)
test.full_report()

Thread Detection Framework Full Report
Open ports found: [22, 80, 443, 8080]
Weak passwords found: ['password', 'hello', 'admin']


---

## 🚀 Section 4 – Challenge (~ 7 minutes)

These are harder problems. Attempt them if you've finished the exercises above.
They require combining multiple OOP concepts at once.

---

### 🔥 Challenge 1 – Responder Hierarchy

Recall `ThreatResponder` (the abstract base class from Concept 4).  

Implement **two more concrete responders:**

**`BlockIPResponder`:**
- Maintains a `blocked_ips` list
- `respond(alert)` — if the alert is a `NetworkIntrusionAlert`, adds the `source_ip` to `blocked_ips` and prints a confirmation; otherwise prints "No IP to block"
- `log_response(alert)` — prints a log entry

**`IncidentTicketResponder`:**
- Maintains a `tickets` list (each ticket is a dict with `id`, `alert_id`, `severity`)
- `respond(alert)` — creates a ticket and appends it
- `log_response(alert)` — prints ticket summary

Then write a function `dispatch(alert, responders)` that takes an alert and a list of responders, and calls `respond()` and `log_response()` on each.

In [ ]:
# Challenge 1 – BlockIPResponder + IncidentTicketResponder + dispatch()

# YOUR CODE HERE


---

### 🔥 Challenge 2 – Design Decision Reflection

Answer in the Markdown cell below.

Consider this scenario:  
> You need to add a `VPNScanner` to the framework. It needs to scan VPN configurations
> for weak encryption settings. It also needs to send email notifications when weak settings are found.

1. Should `VPNScanner` use **inheritance** from `BaseScanner` or **composition**? Why?
2. Should it use `EmailNotifier` through **inheritance** or **composition**? Why?
3. Sketch (in words or pseudocode) how you would design this class.

**Your Answer:**

1. 

2. 

3. 

---

## ✅ Lab Completion Checklist

Before submitting, verify:

- [ ] All warm-up questions answered
- [ ] Exercise 1: `NetworkIntrusionAlert` defined and tested
- [ ] Exercise 2: `print_all_alerts()` demonstrates polymorphism
- [ ] Exercise 3: `PasswordPolicy` + `UserAccount` with composition
- [ ] Exercise 4: `BaseScanner` + `PortScanner` + `PasswordAuditScanner`
- [ ] Exercise 5: `ThreatDetectionFramework` assembled
- [ ] (Bonus) Challenge 1 and/or Challenge 2 attempted

---

**Save your notebook, then submit via GitHub Classroom.**

> *"Good code is its own best documentation."* – Steve McConnell